In [2]:
import sqlite3

DB_NAME = "college.db"

def create_connection():
    conn = sqlite3.connect(DB_NAME)
    return conn


def create_tables(conn):
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS students (
        sid INTEGER PRIMARY KEY,
        sname TEXT NOT NULL
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS courses (
        cid INTEGER PRIMARY KEY,
        cname TEXT NOT NULL,
        credits INTEGER NOT NULL
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS enrollments (
        sid INTEGER NOT NULL,
        cid INTEGER NOT NULL,
        PRIMARY KEY (sid, cid),
        FOREIGN KEY (sid) REFERENCES students(sid),
        FOREIGN KEY (cid) REFERENCES courses(cid)
    );
    """)

    conn.commit()


def insert_sample_data(conn):
    cursor = conn.cursor()

    students = [
        (1001, "Alice Johnson"),
        (1002, "Bob Smith"),
        (1003, "Charlie Brown"),
        (1004, "Diana Lee"),
        (1005, "Ethan Davis")
    ]

    courses = [
        (2001, "Database Systems", 3),
        (2002, "Computer Networks", 3),
        (2003, "Data Structures", 4),
        (2004, "Operating Systems", 4),
        (2005, "Software Engineering", 3)
    ]

    enrollments = [
        (1001, 2001),
        (1001, 2003),
        (1002, 2001),
        (1003, 2002),
        (1003, 2003),
        (1003, 2004),
        (1004, 2002),
        (1004, 2005),
        (1005, 2001),
        (1005, 2005),
        (1005, 2002),
        (1005, 2003),
        (1005, 2004)
    ]

    cursor.executemany("INSERT OR IGNORE INTO students VALUES (?, ?)", students)
    cursor.executemany("INSERT OR IGNORE INTO courses VALUES (?, ?, ?)", courses)
    cursor.executemany("INSERT OR IGNORE INTO enrollments VALUES (?, ?)", enrollments)

    conn.commit()


# -------- NEW FUNCTION TO SHOW STUDENTS --------
def show_students(conn):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT students.sid, students.sname, COUNT(enrollments.cid)
        FROM students
        LEFT JOIN enrollments
        ON students.sid = enrollments.sid
        GROUP BY students.sid, students.sname
    """)

    rows = cursor.fetchall()

    print("\n--- STUDENTS CURRENTLY IN SYSTEM ---")
    print("Student ID | Name | # of Classes")
    print("-----------------------------------")

    for row in rows:
        print(f"{row[0]} | {row[1]} | {row[2]}")
# --------------------------------------


def get_or_create_student(conn):
    cursor = conn.cursor()

    while True:
        # SHOW STUDENTS FIRST
        show_students(conn)

        sid = int(input("\nEnter student ID (-1 for new student): "))

        if sid == -1:
            new_sid = int(input("Enter new student ID: "))
            name = input("Enter student name: ")

            cursor.execute("INSERT INTO students VALUES (?, ?)", (new_sid, name))
            conn.commit()

            print("New student created.")
            return new_sid

        cursor.execute("SELECT * FROM students WHERE sid = ?", (sid,))
        student = cursor.fetchone()

        if student:
            print("Welcome,", student[1])
            return sid
        else:
            print("Student not found.")


def list_courses(conn):
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM courses")
    rows = cursor.fetchall()

    print("\nAll Courses:")
    for row in rows:
        print(row)


def enroll_in_course(conn, sid):
    cursor = conn.cursor()

    cid = int(input("Enter course ID to enroll: "))

    cursor.execute("SELECT * FROM enrollments WHERE sid = ? AND cid = ?", (sid, cid))
    exists = cursor.fetchone()

    if exists:
        print("Already enrolled in that course.")
        return

    cursor.execute("INSERT INTO enrollments VALUES (?, ?)", (sid, cid))
    conn.commit()

    print("Enrollment successful.")


def withdraw_from_course(conn, sid):
    cursor = conn.cursor()

    cid = int(input("Enter course ID to withdraw: "))

    cursor.execute("SELECT * FROM enrollments WHERE sid = ? AND cid = ?", (sid, cid))
    exists = cursor.fetchone()

    if not exists:
        print("You are not enrolled in that course.")
        return

    cursor.execute("DELETE FROM enrollments WHERE sid = ? AND cid = ?", (sid, cid))
    conn.commit()

    print("Withdrawal successful.")
    my_classes(conn, sid)


def search_courses(conn):
    cursor = conn.cursor()

    keyword = input("Enter course name keyword: ")

    cursor.execute(
        "SELECT * FROM courses WHERE cname LIKE ?",
        ('%' + keyword + '%',)
    )

    rows = cursor.fetchall()

    print("\nSearch Results:")
    for row in rows:
        print(row)


def my_classes(conn, sid):
    cursor = conn.cursor()

    cursor.execute("""
        SELECT courses.cid, courses.cname, courses.credits
        FROM courses
        JOIN enrollments
        ON courses.cid = enrollments.cid
        WHERE enrollments.sid = ?
    """, (sid,))

    rows = cursor.fetchall()

    print("\nMy Classes:")

    if rows:
        for row in rows:
            print(row)
    else:
        print("No enrolled classes.")


def main_menu(conn, sid):
    while True:
        print("\n--- MAIN MENU ---")
        print("L - List Courses")
        print("E - Enroll")
        print("W - Withdraw")
        print("S - Search")
        print("M - My Classes")
        print("X - Exit")

        choice = input("Choose option: ").upper()

        if choice == "L":
            list_courses(conn)

        elif choice == "E":
            enroll_in_course(conn, sid)

        elif choice == "W":
            withdraw_from_course(conn, sid)

        elif choice == "S":
            search_courses(conn)

        elif choice == "M":
            my_classes(conn, sid)

        elif choice == "X":
            print("Goodbye.")
            break

        else:
            print("Invalid option.")


def main():
    conn = create_connection()
    create_tables(conn)
    insert_sample_data(conn)

    sid = get_or_create_student(conn)

    main_menu(conn, sid)

    conn.close()


if __name__ == "__main__":
    main()


--- STUDENTS CURRENTLY IN SYSTEM ---
Student ID | Name | # of Classes
-----------------------------------
1001 | Alice Johnson | 2
1002 | Bob Smith | 2
1003 | Charlie Brown | 3
1004 | Diana Lee | 2
1005 | Ethan Davis | 5

Enter student ID (-1 for new student): -1
Enter new student ID: -1


KeyboardInterrupt: Interrupted by user